In [33]:
# src/kooplearn/metrics.py
# This file has been modified from the original version
# to add three new diagnostic metrics from Kostic et al. 2023:
# - metric distortion
# - spectral bias
# - spurious eigenvalue count

import numpy as np

from kooplearn._linalg import covariance, spd_neg_pow


# This function is the same as the original version.
def directed_hausdorff_distance(pred: np.ndarray, reference: np.ndarray):
    """One-sided directed Hausdorff distance between two 1D sets. Useful for computing distances
    between estimated eigenvalues

    Calculates the directed Hausdorff distance
    :math:`\\vec{H}(A, B) = \\max_{a \\in A} \\min_{b \\in B} \\|a - b\\|_p` where :math:`A` is the
    set of points in "pred" and :math:`B` is the set of points in "reference". The current
    implementation uses the :math:`L_1` norm: :math:`\\|a - b\\|_1 = |a - b|`.

    Parameters
    ----------
    pred : numpy.ndarray
        The set of predicted points :math:`A`. Must be a 1D array.
    reference : numpy.ndarray
        The set of reference points :math:`B`. Must be a 1D array.

    Returns
    -------
    float
        The directed Hausdorff distance between "pred"and "reference".

    Raises
    ------
    AssertionError
        If "pred" or "reference" are not 1-dimensional arrays.

    Examples
    --------

    .. code-block:: python

        import numpy as np
        from kooplearn.metrics import directed_hausdorff_distance
        pred = np.array([1, 5, 6])
        reference = np.array([2, 4, 7])
        directed_hausdorff_distance(pred, reference)
        # Will print np.float64(1.0)
    """
    pred = np.asanyarray(pred)
    reference = np.asanyarray(reference)
    assert pred.ndim == 1
    assert reference.ndim == 1

    distances = np.zeros((pred.shape[0], reference.shape[0]), dtype=np.float64)
    for pred_idx, pred_pt in enumerate(pred):
        for reference_idx, reference_pt in enumerate(reference):
            distances[pred_idx, reference_idx] = np.abs(pred_pt - reference_pt)
    hausdorff_dist = np.max(np.min(distances, axis=1))
    return hausdorff_dist


# Edits from Kostic et al. 2023 inserted below.
# ──────────────────────────────────────────────────────────────────
# 1.  Operator norm error
# ──────────────────────────────────────────────────────────────────


def operator_norm_error(true_operator: np.ndarray, estimated_operator: np.ndarray):
    r"""Operator norm error proxy for a Koopman estimator.

    Computes the operator norm discrepancy between the true action
    :math:`A_\pi S` and the estimated action :math:`S \widehat{G}`:

    .. math::

        \mathcal{E}(\widehat{G}) := \|A_\pi S - S \widehat{G}\|.

    Since "kooplearn" does not currently expose :math:`A_\pi` or the embedding
    operator :math:`S` explicitly, this function works with their actions on a
    common finite-dimensional representation. In practice, the caller should pass
    matrices or vectors representing the two quantities to be compared.

    Parameters
    ----------
    true_operator : numpy.ndarray
        Array representing the reference action :math:`A_\pi S`.
    estimated_operator : numpy.ndarray
        Array representing the estimated action :math:`S \widehat{G}`.
        Must have the same shape as "true_operator".

    Returns
    -------
    float
        The spectral norm of the discrepancy if the inputs are matrices, or the
        Euclidean norm if the inputs are vectors.

    Raises
    ------
    ValueError
        If the inputs do not have the same shape.

    Notes
    -----
    For 1D arrays, this function returns the Euclidean norm.
    For 2D arrays this function returns the spectral norm,
    the largest singular value of the difference.

    This function is most useful when a trusted reference operator is
    available, for example in synthetic experiments or benchmark problems.
    It should not be interpreted as automatically estimating the theoretical
    quantity :math:`\|A_\pi S - S\widehat{G}\|` from data alone.

    Examples
    --------

    .. code-block:: python

        import numpy as np
        from kooplearn.metrics import operator_norm_error

        A_pi_S = np.array([[1.0, 0.0], [0.0, 0.8]])
        S_Ghat = np.array([[0.9, 0.1], [0.0, 0.75]])
        err = operator_norm_error(A_pi_S, S_Ghat)
        print(err)
    """
    true_operator = np.asanyarray(true_operator)
    estimated_operator = np.asanyarray(estimated_operator)

    if true_operator.shape != estimated_operator.shape:
        raise ValueError(
            "true_operator and estimated_operator must have the same "
            f"shape, got {true_operator.shape} and "
            f"{estimated_operator.shape}."
        )

    diff = true_operator - estimated_operator
    if diff.ndim == 1:
        return float(np.linalg.norm(diff))
    return float(np.linalg.norm(diff, ord=2))


# ──────────────────────────────────────────────────────────────────
# 2.  Metric distortion
# ──────────────────────────────────────────────────────────────────


def metric_distortion(eigenfunctions: np.ndarray, covariance: np.ndarray):
    r"""Metric distortion of a vector under the observation map.

    Computes the empirical metric distortion:

    .. math::
        \widehat{\eta}_i :=
        \frac{\|\widehat{\psi}_i\|}{\sqrt{
        \langle \widehat{C}\widehat{\psi}_i, \widehat{\psi}_i \rangle
        }}.

    Parameters
    ----------
    eigenfunction : numpy.ndarray, shape (n_features,)
        Empirical eigenfunction :math:`\widehat{\psi}_i`.
    covariance : numpy.ndarray, shape (n_features, n_features)
        Empirical covariance operator :math:`\widehat{C}` represented as a matrix.

    Returns
    -------
    float
        The empirical metric distortion :math:`\widehat{\eta}_i`.

    Raises
    ------
    ValueError
        If "eigenfunction" is not one-dimensional.
    ValueError
        If "covariance" is not a square matrix compatible with "eigenfunction".
    ValueError
        If :math:`\langle \widehat{C}\widehat{\psi}_i, \widehat{\psi}_i \rangle \leq 0`.

    Notes
    -----
    This is a practical diagnostic rather than the theorem-level quantity.

    The denominator is computed as:
    :math:`\langle \widehat{C}\widehat{\psi}_i, \widehat{\psi}_i \rangle`,
    where :math:`\widehat{C}` is the empirical covariance operator
    and :math:`\widehat{\psi}_i` is the empirical eigenfunction.
    Only its real part is retained, which is appropriate when
    "covariance" is Hermitian up to numerical error.

    Examples
    --------

    .. code-block:: python

        import numpy as np
        from kooplearn.metrics import metric_distortion

        psi_hat = np.array([1.0, 0.0])
        C_hat = np.array([[2.0, 0.0], [0.0, 1.0]])
        eta_hat = metric_distortion(psi_hat, C_hat)
        print(eta_hat)
    """
    eigenfunction = np.asanyarray(eigenfunctions)
    covariance = np.asanyarray(covariance)

    if eigenfunction.ndim != 1:
        raise ValueError(f"eigenfunction must be 1D, got array with shape {eigenfunction.shape}.")
    if covariance.ndim != 2 or covariance.shape[0] != covariance.shape[1]:
        raise ValueError(
            f"covariance must be a square 2D array, got array with shape {covariance.shape}."
        )
    if covariance.shape[0] != eigenfunction.shape[0]:
        raise ValueError(
            "covariance and eigenfunction have incompatible shapes: "
            f"{covariance.shape} and {eigenfunction.shape}."
        )

    numerator = np.linalg.norm(eigenfunction)
    quadratic_form = np.real(np.vdot(eigenfunction, covariance @ eigenfunction))

    if quadratic_form <= 0.0:
        raise ValueError("The quadratic form <C_hat psi_hat, psi_hat> must be strictly positive.")

    return float(numerator / np.sqrt(quadratic_form))


# ──────────────────────────────────────────────────────────────────
# 3.  Spectral bias
# ──────────────────────────────────────────────────────────────────
# Helpers for truncation term
# ===========================
def center_matrix(K):
    n = K.shape[0]
    H = np.eye(n) - np.ones((n, n)) / n
    return H @ K @ H


def spectral_bias(eigenfunctions: np.ndarray,
    covariance: np.ndarray,
    truncation: float,
) -> tuple[float, float]:
    r"""Empirical spectral bias for a Koopman estimator.

    Computes the empirical spectral bias s-hat-i from eq 18:
    (generalised form here)

    .. math::
        \hat{s}_i(\widehat{G}_{r,\gamma})
        := \widehat{\eta}_i \, \rho_{r+1}(\widehat{G}_{r,\gamma}),

    where :math:`\widehat{\eta}_i` is the metric distortion and
    :math:`\rho_{r+1}(\widehat{G}_{r,\gamma})` is the estimator-specific
    truncation quantity controlling the rank-\((r+1)\) contribution.

    Parameters
    ----------
    eigenfunction : numpy.ndarray, shape (n_features,)
        Empirical eigenfunction :math:`\widehat{\psi}_i`.
    covariance : numpy.ndarray, shape (n_features, n_features)
        Empirical covariance operator :math:`\widehat{C}` represented as a matrix.
    truncation : float
        Estimator-specific quantity
        :math:`\rho_{r+1}(\widehat{G}_{r,\gamma})`.

    Returns
    -------
    float
        The empirical spectral bias
        :math:`\hat{s}_i(\widehat{G}_{r,\gamma})`.


    Notes
    -----
    This function is intentionally algorithm-agnostic. For example, in the
    formulas of Kostic et al. (2023), one may take

    - :math:`\rho_{r+1}(\widehat{G}^{\mathrm{PCR}}_{r,\gamma})
      = \sigma_{r+1}(\widehat{C})`, (i.e., truncation term is the (r+1)-st singular
      value of the empirical covariance,)
    - :math:`\rho_{r+1}(\widehat{G}^{\mathrm{RRR}}_{r,\gamma})
      = \sigma_{r+1}(\widehat{C}^{-1/2}\widehat{T})`.

    Examples
    --------

    .. code-block:: python

        from kooplearn.metrics import spectral_bias, pcr_truncation, kdmd_truncation

        values, functions = model.eig(eval_right_on=x)

        K = model.kernel(X_train, X_train)   # if callable kernel
        trunc_pcr = pcr_truncation(K, r)
        trunc_kdmd = kdmd_truncation(K, r)

        # For each eigenfunction:
        bias_pcr = spectral_bias(functions, covariance=K, truncation=trunc_pcr)
        bias_kdmd = spectral_bias(functions, covariance=K, truncation=trunc_kdmd)
    """
    eta = metric_distortion(eigenfunctions, covariance)
    s_hat = eta * truncation
    return s_hat, eta


# ──────────────────────────────────────────────────────────────────
# 4.  Spurious eigenvalue count
# ──────────────────────────────────────────────────────────────────


def spurious_eigenvalues(
    estimated_eigenvalues: np.ndarray,
    reference_eigenvalues: np.ndarray,
    delta: float,
):
    r"""Count estimated eigenvalues that are farther than "delta" from a reference set.

    Computes the number of potentially spurious eigenvalues,

    .. math::
        N_{\mathrm{spur}}(\delta)
        =
        \#\left\{
            \widehat{\lambda}_j :
            \operatorname{dist}(\widehat{\lambda}_j, \sigma(A)) > \delta
        \right\},

    where the reference set (i.e., the true spectrum) is supplied explicitly,
    via "reference_eigenvalues".

    Parameters
    ----------
    estimated_eigenvalues : numpy.ndarray
        One-dimensional array of estimated eigenvalues.
    reference_eigenvalues : numpy.ndarray
        One-dimensional array of trusted or reference eigenvalues.
    delta : float
        Distance threshold. Estimated eigenvalues whose distance from the
        reference set exceeds "delta" are counted as spurious.

    Returns
    -------
    int
        Number of estimated eigenvalues farther than "delta" from the
        reference set.

    Raises
    ------
    ValueError
        If the inputs are not one-dimensional or if "delta" is not positive.

    Notes
    -----
    This is a practical diagnostic rather than a theorem-level quantity.
    It is most useful when a trusted reference spectrum is available,
    for example from an analytically solvable problem, a high-resolution
    computation, or a benchmark estimator.

    Examples
    --------

    .. code-block:: python

        import numpy as np
        from kooplearn.metrics import spurious_eigenvalues

        estimated = np.array([0.9 + 0.0j, 0.5 + 0.2j, -1.5 + 0.0j])
        reference = np.array([0.9 + 0.0j, 0.5 + 0.2j])
        spurious_eigenvalues(estimated, reference, delta=0.1)
    """
    estimated_eigenvalues = np.asanyarray(estimated_eigenvalues)
    reference_eigenvalues = np.asanyarray(reference_eigenvalues)

    if estimated_eigenvalues.ndim != 1:
        raise ValueError(
            f"estimated_eigenvalues must be a 1D array, got shape {estimated_eigenvalues.shape}."
        )
    if reference_eigenvalues.ndim != 1:
        raise ValueError(
            f"reference_eigenvalues must be a 1D array, got shape {reference_eigenvalues.shape}."
        )
    if delta <= 0.0:
        raise ValueError(f"delta must be strictly positive, got {delta}.")

    # dist(λ̂_j, σ(A)) = min_k |λ̂_j - σ_k|  in ℂ  (L1 on complex plane = |·|)
    # Expected shape: (r, 1) - (1, s) → (r, s)
    distances = np.abs(
        estimated_eigenvalues[:, np.newaxis] - reference_eigenvalues[np.newaxis, :]
    )  # shape (r, s)
    min_distances = np.min(distances, axis=1)  # shape (r,)
    return int(np.sum(min_distances > delta))


In [12]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.integrate import romb
from tqdm import tqdm

from kooplearn._utils import stable_topk
from kooplearn.datasets import make_prinz_potential
from kooplearn.kernel import KernelRidge

# from kooplearn.metrics import kdmd_truncation, rrr_truncation, spectral_bias

n_train_samples = 5001
n_show = 4
n_components = 10
gamma = 1.0
sigma = 2.0


def prinz_potential(x):
    return 4 * (
        x**8
        + 0.8 * np.exp(-80 * (x**2))
        + 0.2 * np.exp(-80 * ((x - 0.5) ** 2))
        + 0.5 * np.exp(-40 * ((x + 0.5) ** 2))
    )


def compute_boltzmann_density(x, gamma, sigma):
    beta = 2 * gamma / (sigma**2)
    pdf = np.exp(-beta * prinz_potential(x))
    total_mass = romb(pdf, dx=x[1] - x[0])
    return pdf / total_mass


def fit_and_estimate(reduced_rank, x, density, random_state):
    subsample = 100  # long trajectories for sampling Boltzmann distribution

    # Parameters of Langevin equation
    gamma = 1.0  # damping coefficient
    sigma = 2.0  # noise coefficient, proportional to temperature
    data = make_prinz_potential(
        X0=0, n_steps=int(7e5), gamma=gamma, sigma=sigma, random_state=random_state
    )

    # Sample new trajectories
    data = data.iloc[
        ::subsample  # don't need all data
    ]

    data = data[:n_train_samples]

    # Model definition
    model = KernelRidge(
        n_components=5,
        reduced_rank=reduced_rank,
        gamma=12.5,
        kernel="rbf",
        alpha=1e-6,
        random_state=random_state,
    )

    # Fit and estimate eigenfunctions
    model.fit(data)  # fit transfer op model
    values, functions = model.eig(
        eval_right_on=x
    )  # (right) eigenvalue estimation, evaluate on array x
    sort_perm = np.flip(np.argsort(np.abs(values)))  # Order decreasingly
    values, functions = values[sort_perm], functions[:, sort_perm]
    functions = normalize_eigenfunctions(functions, x, density)
    return values, functions


def normalize_eigenfunctions(functions, x, density):
    abs2_eigfun = (np.abs(functions) ** 2).T  # f(x)**2
    abs2_eigfun *= density  # Compute the norm with respect to the Boltzmann measure.
    dx = x[1] - x[0]
    funcs_norm = np.sqrt(romb(abs2_eigfun, dx=dx, axis=1))  # Norms
    functions *= funcs_norm**-1.0  # Normalize
    return functions


def analyse_bias(records, out_prefix="spectral_bias"):
    df = pd.DataFrame(records).copy()

    if "eigenvalue_error" not in df.columns and {"true_eig", "est_eig"}.issubset(
        df.columns
    ):
        df["eigenvalue_error"] = np.abs(df["est_eig"] - df["true_eig"])

    summary = df.groupby(["kernel", "method", "eigenfunction_id"], as_index=False).agg(
        n=("spectral_bias", "size"),
        bias_mean=("spectral_bias", "mean"),
        bias_std=("spectral_bias", "std"),
        err_mean=("eigenvalue_error", "mean"),
        err_std=("eigenvalue_error", "std"),
        dist_mean=("metric_distortion", "mean")
        if "metric_distortion" in df.columns
        else ("spectral_bias", "mean"),
        trunc_mean=("truncation", "mean")
        if "truncation" in df.columns
        else ("spectral_bias", "mean"),
    )

    corr_rows = []
    for (kernel, method, eig_id), g in df.groupby(
        ["kernel", "method", "eigenfunction_id"]
    ):
        if len(g) >= 2:
            pear = g["spectral_bias"].corr(g["eigenvalue_error"], method="pearson")
            spear = g["spectral_bias"].corr(g["eigenvalue_error"], method="spearman")
        else:
            pear = np.nan
            spear = np.nan
        corr_rows.append(
            {
                "kernel": kernel,
                "method": method,
                "eigenfunction_id": eig_id,
                "pearson": pear,
                "spearman": spear,
                "n": len(g),
            }
        )
    corr_df = pd.DataFrame(corr_rows)

    summary.to_csv(f"{out_prefix}_summary.csv", index=False)
    corr_df.to_csv(f"{out_prefix}_correlations.csv", index=False)
    df.to_csv(f"{out_prefix}_records.csv", index=False)

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for (kernel, method), g in df.groupby(["kernel", "method"]):
        ax.scatter(
            g["spectral_bias"],
            g["eigenvalue_error"],
            s=20,
            alpha=0.7,
            label=f"{kernel} / {method}",
        )
    ax.set_xlabel("Spectral bias")
    ax.set_ylabel("Eigenvalue error")
    ax.legend(frameon=False, fontsize=8)
    ax.set_title("Spectral bias vs eigenvalue error")
    fig.tight_layout()
    fig.savefig(f"{out_prefix}_scatter.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    return df, summary, corr_df


In [17]:
import torch
from torch.utils.data import DataLoader, TensorDataset

device = "cuda" if torch.cuda.is_available() else "cpu"

n_train_samples = 5000
n_val_samples = 1000

subsample = 100
batch_size = 512

# Experiment hyperparameters
learning_rate = 1e-2
opt = torch.optim.AdamW
num_epochs = 200
layer_dims = [32, 64, 32]
latent_dim = 32
random_state = 42

data = make_prinz_potential(
    X0=0, n_steps=int(7e5), gamma=gamma, sigma=sigma, random_state=0
)
data = data.iloc[
    ::subsample
]  # ...but we don't need all the data to estimate the eigenfunctions

train_data = torch.from_numpy(data[:n_train_samples].values).float()
val_data = torch.from_numpy(data[-n_val_samples:].values).float()

# Creating PyTorch TensorDatasets
train_ds = TensorDataset(train_data[:-1], train_data[1:])
val_ds = TensorDataset(val_data[:-1], val_data[1:])

# Creating DataLoaders
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=len(val_ds), shuffle=False)

In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SimpleMLP(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int, layer_dims=(128, 128)):
        super().__init__()
        dims = [input_dim, *layer_dims, latent_dim]
        layers = []
        for i in range(len(dims) - 2):
            layers += [nn.Linear(dims[i], dims[i + 1]), nn.ReLU()]
        layers += [nn.Linear(dims[-2], dims[-1])]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class FeatureMap(nn.Module):
    def __init__(
        self,
        input_dim: int,
        M: int,
        kind: str = "mlp",
        r: int | None = None,
        normalize_latents: bool = False,
        layer_dims=(128, 128),
    ):
        super().__init__()
        self.input_dim = input_dim
        self.M = M
        self.kind = kind
        self.r = r
        self.normalize_latents = normalize_latents

        if kind != "mlp":
            raise ValueError(f"Unsupported kind={kind!r}; this block currently implements kind='mlp' only.")

        self.backbone = SimpleMLP(input_dim=input_dim, latent_dim=M, layer_dims=layer_dims)
        self.lin = nn.Linear(M, M, bias=False)

    def encode(self, X: torch.Tensor) -> torch.Tensor:
        Z = self.backbone(X)
        if self.normalize_latents:
            Z = F.normalize(Z, dim=-1)
        return Z

    def forward(self, X: torch.Tensor, lagged: bool = False) -> torch.Tensor:
        Z = self.encode(X)
        if lagged:
            Z = self.lin(Z)
        return Z


@torch.no_grad()
def compute_operators(
    feature_map: FeatureMap,
    X: torch.Tensor,
    Y: torch.Tensor,
    center: bool = False,
    apply_lagged_to_Y: bool = True,
):
    Phi_X = feature_map(X)
    Phi_Y = feature_map(Y, lagged=apply_lagged_to_Y)

    if center:
        Phi_X = Phi_X - Phi_X.mean(dim=0, keepdim=True)
        Phi_Y = Phi_Y - Phi_Y.mean(dim=0, keepdim=True)

    n = Phi_X.shape[0]

    Cx = covariance(Phi_X)          # empirical feature covariance
    Cy = covariance(Phi_Y)          # lagged/response feature covariance
    T = covariance(Phi_X,Phi_Y)         # cross-covariance / transfer operator ingredient

    return {
        "Phi_X": Phi_X,
        "Phi_Y": Phi_Y,
        "C": Cx,
        "C_x": Cx,
        "C_y": Cy,
        "T": T,
        "n": n,
    }


def pcr_truncation(C, r: int) -> torch.Tensor:
    C = torch.as_tensor(C, dtype=torch.float32, device=device)
    svals = torch.linalg.svdvals(C)
    if r >= len(svals):
        return torch.tensor(0.0, device=C.device, dtype=C.dtype)
    return svals[r]   # (r+1)-st singular value


def rrr_truncation(C, T, r: int, eps: float = 1e-12) -> torch.Tensor:
    C = torch.as_tensor(C, dtype=torch.float32, device=device)
    T = torch.as_tensor(T, dtype=torch.float32, device=device)

    evals, evecs = torch.linalg.eigh(C)
    evals = torch.clamp(evals, min=eps)
    C_inv_sqrt = evecs @ torch.diag(evals.rsqrt()) @ evecs.T

    A = C_inv_sqrt @ T
    svals = torch.linalg.svdvals(A)
    if r >= len(svals):
        return torch.tensor(0.0, device=C.device, dtype=C.dtype)
    return svals[r]   # (r+1)-st singular value

def kdmd_truncation(C, r: int) -> torch.Tensor:
    C = torch.as_tensor(C, dtype=torch.float32, device=device)
    svals = torch.linalg.svdvals(C)
    if r >= len(svals):
        return torch.tensor(0.0, device=C.device, dtype=C.dtype)
    return svals[r]   # (r+1)-st singular value


In [ ]:
x = np.linspace(-2, 2, 2048 + 1)
density = compute_boltzmann_density(x, gamma, sigma)
data = make_prinz_potential(X0=0, n_steps=int(7e5), gamma=gamma, sigma=sigma)

X = data[["x"]].to_numpy()
X_train = X[:-1]
Y_train = X[1:]

density = np.asarray(density).reshape(-1)

latent_dim = 10


dx = np.diff(x)
w = np.empty_like(x, dtype=float)
w[1:-1] = 0.5 * (dx[:-1] + dx[1:])
w[0] = 0.5 * dx[0]
w[-1] = 0.5 * dx[-1]

weight_matrix = np.diag(w * density)

records = []

X_train = torch.from_numpy(X_train).float()
Y_train = torch.from_numpy(Y_train).float()

for method, reduced_rank in zip(
    ["Principal Components (kDMD)", "Reduced Rank"], [False, True]
):
    for i in tqdm(range(10)):
        x = np.asarray(x).reshape(-1, 1)
        values, functions = fit_and_estimate(reduced_rank, x, density, i)
        # only top eigenfunctions
        vals, idx = stable_topk(values, k_max=n_show)
        r = len(vals)
        top_eigs = functions[:, idx]

        if method == "Principal Components (kDMD)":
            feature_map = FeatureMap(
            input_dim=X_train.shape[1],
            M=latent_dim,
            kind="mlp",
            r= r,
            normalize_latents=False,
            ).to(device)

            ops = compute_operators(
                feature_map=feature_map,
                X=X_train,
                Y=Y_train,
                center=False,
                apply_lagged_to_Y=True,
            )

            Phi_X = ops["Phi_X"]
            Phi_Y = ops["Phi_Y"]
            C = ops["C"]
            trunc_kdmd = kdmd_truncation(C, r=r)

            for eig_id in range(min(2, len(idx))):
                eigfun = functions[:, idx[eig_id]]
                bias, distortion = spectral_bias(
                    eigenfunctions=eigfun,
                    covariance=C,
                    truncation=trunc_kdmd,
                )
                records.append({
                    "kernel": "rbf",
                    "method": method,
                    "trial": i,
                    "eigenfunction_id": eig_id + 1,
                    "spectral_bias": float(bias),
                    "eigenvalue_error": float(np.abs(values[eig_id] - top_eigs[eig_id])),
                    "metric_distortion": float(distortion),
                    "truncation": float(trunc_kdmd),
                    "est_eig": float(values[eig_id]),
                    "true_eig": float(top_eigs[eig_id]),
                })

        else:
            feature_map = FeatureMap(
            input_dim=X_train.shape[1],
            M=latent_dim,
            kind="mlp",
            r= r,
            normalize_latents=False,
            ).to(device)

            ops = compute_operators(
                feature_map=feature_map,
                X=X_train.to(device),
                Y=Y_train.to(device),
                center=False,
                apply_lagged_to_Y=True,
            )

            Phi_X = ops["Phi_X"]
            Phi_Y = ops["Phi_Y"]
            C = ops["C"]
            T = ops["T"]
            trunc_rrr = rrr_truncation(C, T, r=r)

            for eig_id in range(min(2, len(idx))):
                eigfun = functions[:, idx[eig_id]]
                bias, distortion = spectral_bias(
                    eigenfunctions=eigfun,
                    covariance=C,
                    truncation=trunc_rrr,
                )
                records.append({
                    "kernel": "rbf",
                    "method": method,
                    "trial": i,
                    "eigenfunction_id": eig_id + 1,
                    "spectral_bias": float(bias),
                    "eigenvalue_error": float(np.abs(values[eig_id] - top_eigs[eig_id])),
                    "metric_distortion": float(distortion),
                    "truncation": float(trunc_rrr),
                    "est_eig": float(values[eig_id]),
                    "true_eig": float(top_eigs[eig_id]),
                })

df, summary, corr_df = analyse_bias(records)

  0%|          | 0/10 [00:06<?, ?it/s]


ValueError: eigenfunction must be 1D, got array with shape (2049, 5).